In [1]:
# Online Retail II - Data Cleaning & Feature Engineering

**Objective:** Clean the raw Online Retail II transactional data and produce 2 analysis datasets for use in Tableau

1. 'transaction_clean.csv' - line-item level transactions
2. 'customers_clean.csv' - customer-level aggregated metrics

**Source:** UCI Machine Learning Repository- Online Retail II
**Period covered:** 2010- 2011

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format','{:,.2f}' .format)

## 1. Load the data

In [4]:
# File is in the same folder as the notebook for now
df_2009 = pd.read_excel('Online_retail.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('Online_retail.xlsx', sheet_name='Year 2010-2011')

print(f"2009-2010 sheet: {df_2009.shape[0]:,} rows, {df_2009.shape[1]} columns")
print(f"2010-2011 sheet: {df_2010.shape[0]:,} rows, {df_2010.shape[1]} columns")

df = pd.concat([df_2009, df_2010], ignore_index=True)
print(f"\nCombined dataset: {df.shape[0]:,} rows")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

2009-2010 sheet: 525,461 rows, 8 columns
2010-2011 sheet: 541,910 rows, 8 columns

Combined dataset: 1,067,371 rows
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [5]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [7]:
df.describe(include = 'all')

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,"1,067,371.00",1067371,1062989,"1,067,371.00",1067371,"1,067,371.00","824,364.00",1067371
unique,"53,628.00",5305,5698,NaN,NaN,NaN,NaN,43
top,"537,434.00",85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,"1,350.00",5829,5918,NaN,NaN,NaN,NaN,981330
mean,NaN,NaN,NaN,9.94,2011-01-02 21:13:55.394028544,4.65,"15,324.64",NaN
min,NaN,NaN,NaN,"-80,995.00",2009-12-01 07:45:00,"-53,594.36","12,346.00",NaN
25%,NaN,NaN,NaN,1.00,2010-07-09 09:46:00,1.25,"13,975.00",NaN
50%,NaN,NaN,NaN,3.00,2010-12-07 15:28:00,2.10,"15,255.00",NaN
75%,NaN,NaN,NaN,10.00,2011-07-22 10:23:00,4.15,"16,797.00",NaN
max,NaN,NaN,NaN,"80,995.00",2011-12-09 12:50:00,"38,970.00","18,287.00",NaN


In [8]:
missing = df.isnull().sum()
missing_pct = (missing/len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df[missing_df['missing_count']> 0].sort_values('missing_count', ascending = False)

,missing_count,missing_pct
Customer ID,243007,22.77
Description,4382,0.41


In [9]:
# Cancelled orders have invoice numbers starting with 'C'
df['Invoice'] = df['Invoice'].astype(str)
cancelled = df[df['Invoice'].str.startswith('C')]
print(f"Cancelled Transactions: {len(cancelled):,} ({len(cancelled)/len(df)*100:.2f}%)")

# Negative quantities not in cancellations - usually data errors
neg_qty_non_cancel = df[(df['Quantity']<0) & (~df['Invoice'].str.startswith('C'))]
print(f"Negative quantity (non-cancelled): {len(cancelled):,}")

#Zero or negative prices
bad_price = df[df['Price']<=0]
print(f"Transactions with zero/negative price: {len(bad_price):,}")

#Duplicates
dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {dupes:,}")

Cancelled Transactions: 19,494 (1.83%)
Negative quantity (non-cancelled): 19,494
Transactions with zero/negative price: 6,207
Exact duplicate rows: 34,335


In [10]:
# Standard product codes are 5-6 characters, mostly numeric, sometimes with a letter suffix
df['StockCode'] = df['StockCode'].astype(str)
non_standard = df[~df['StockCode'].str.match(r'^\d{5,6}[A-Za-z]?$')]
print(f"Non-standard stock codes: {non_standard['StockCode'].nunique()} unique codes")
print(f"Total rows with non-standard codes: {len(non_standard):,}\n")
print("Top 15 non-standard codes by frequency:")
print(non_standard['StockCode'].value_counts().head(15))

Non-standard stock codes: 68 unique codes
Total rows with non-standard codes: 7,466

Top 15 non-standard codes by frequency:
StockCode
POST            2122
DOT             1446
M               1421
15056BL          923
C2               282
79323LP          232
D                177
79323GR          123
S                104
BANK CHARGES     102
15056bl           93
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
Name: count, dtype: int64


| Issue |Count | Decision |
|-------|---------------|----------|
| Missing Customer ID | 22.77% | Keep for revenue/product analysis; exclude from customer-level analysis |
| Missing Description | 19494 | Drop (likely system artifacts) |
| Cancelled orders (Invoice starts with 'C') | 1.83% | Move to separate `returns` df; exclude from main revenue but track return rate as a KPI |
| Negative quantity (non-cancelled) | 19494 | Drop — likely data entry errors |
| Zero/negative prices | 6207 | Drop — invalid transactions |
| Non-product StockCodes | 68 unique codes | Flag with a category column; keep for revenue, exclude from product analysis |
| Exact duplicate rows | 34335 | Drop |

In [11]:
# Confirm Country is text and clean
print("=== Country ===")
print("Type:", df['Country'].dtype)
print("Unique:", df['Country'].nunique())
print(df['Country'].value_counts().head(5))

# Confirm Customer ID column name and type
print("\n=== Customer ID ===")
print("Type:", df['Customer ID'].dtype)
print("Sample non-null values:", df['Customer ID'].dropna().head(3).tolist())

# Confirm what's in suspicious descriptions
print("\n=== Suspicious Descriptions ===")
print(df[df['Description'].astype(str).str.contains(r'^\d+$|^\?$', na=False, regex=True)]['Description'].value_counts().head(10))

=== Country ===
Type: object
Unique: 43
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Name: count, dtype: int64

=== Customer ID ===
Type: float64
Sample non-null values: [13085.0, 13085.0, 13085.0]

=== Suspicious Descriptions ===
Description
?        92
21494     1
22719     1
22467     1
20713     1
Name: count, dtype: int64


## 4. Cleaning Operations

We now apply the cleaning decisions from the data quality summary. We track row counts at each step so the impact of every operation is visible and reversible as needed

In [12]:
print(f"Starting rows: {len(df):,}")
print(f"Starting unique customers: {df['Customer ID'].nunique():,}")
print(f"Starting unique products: {df['StockCode'].nunique():,}")

Starting rows: 1,067,371
Starting unique customers: 5,942
Starting unique products: 5,305


In [13]:
df_clean = df.copy()

# Drop rows with missing Description
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Description'])
print(f"Dropped {before-len(df_clean):,} rows with missing Description")

# Drop exact duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean):,} exact duplicate rows")

print(f"\nRemaining rows: {len(df_clean):,}")

Dropped 4,382 rows with missing Description
Dropped 34,228 exact duplicate rows

Remaining rows: 1,028,761


In [14]:
# Cancellations and returns kept separately - useful for return rate KPI
returns = df_clean[df_clean['Invoice'].str.startswith('C')].copy()
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C')].copy()

print(f"Returns separated: {len(returns):,} rows")
print(f"Main transactions remaining: {len(df_clean):,}")

Returns separated: 19,104 rows
Main transactions remaining: 1,009,657


In [15]:
# Negative quantities (these are non-cancellation negatives - data errors)
before = len(df_clean)
df_clean = df_clean[df_clean['Quantity']>0]
print(f"Dropped {before-len(df_clean):,} rows with negative/zero quantity")

#Zero or negative prices
before = len(df_clean)
df_clean = df_clean[df_clean['Price']>0]
print(f"Dropped {before - len(df_clean):,} rows with zero/negative price")

print(f"\nRemaining rows: {len(df_clean):,}")

Dropped 760 rows with negative/zero quantity
Dropped 984 rows with zero/negative price

Remaining rows: 1,007,913


## 5. Feature Engineering

We create calculated columns that the dashboard will need. Revenue, time based dimensions, and a flag distinguishing real products from administrative line items (postage, manual adjustments etc)

In [16]:
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

# Time based feature
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

# Flag whether each row is a real product or an administrative line item
df_clean['IsProduct'] = df_clean['StockCode'].str.match(r'^\d{5,6}[A-Za-z]?$')

# Convert CustomerID to clean string (drop the .0 from float representation)
df_clean['Customer ID'] = df_clean['Customer ID'].apply(
    lambda x: str(int(x)) if pd.notna(x) else None
)

print("New columns added:", ['Revenue', 'Year', 'Month', 'YearMonth', 'DayOfWeek', 'Hour', 'IsProduct'])
print(f"\nProduct rows: {df_clean['IsProduct'].sum():,}")
print(f"Non-product rows (postage, fees, etc.): {(~df_clean['IsProduct']).sum():,}")
df_clean.head()

New columns added: ['Revenue', 'Year', 'Month', 'YearMonth', 'DayOfWeek', 'Hour', 'IsProduct']

Product rows: 1,002,004
Non-product rows (postage, fees, etc.): 5,909


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,YearMonth,DayOfWeek,Hour,IsProduct
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,2009,12,2009-12,Tuesday,7,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009,12,2009-12,Tuesday,7,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009,12,2009-12,Tuesday,7,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,2009,12,2009-12,Tuesday,7,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,2009,12,2009-12,Tuesday,7,True


In [18]:
print("=== Final dataset health ===")
print(f"Total rows: {len(df_clean):,}")
print(f"Total revenue: £{df_clean['Revenue'].sum():,.2f}")
print(f"Date range: {df_clean['InvoiceDate'].min().date()} to {df_clean['InvoiceDate'].max().date()}")
print(f"Unique customers (excluding nulls): {df_clean['Customer ID'].nunique():,}")
print(f"Unique invoices: {df_clean['Invoice'].nunique():,}")
print(f"Unique products: {df_clean[df_clean['IsProduct']]['StockCode'].nunique():,}")
print(f"Countries: {df_clean['Country'].nunique()}")

=== Final dataset health ===
Total rows: 1,007,913
Total revenue: £20,476,260.45
Date range: 2009-12-01 to 2011-12-09
Unique customers (excluding nulls): 5,878
Unique invoices: 40,077
Unique products: 4,873
Countries: 43


## 6. Build customer-level dataset

We aggregate the cleaned transactions to one row per customer. This is what we'll use for customer segmentation, RFM analysis, and any chart that asks "how do customers behave?" rather than "what happened in this transaction?"

In [19]:
# Filter to rows where Customer ID exists - we can't aggregate by missing IDs
df_customers_source = df_clean[df_clean['Customer ID'].notna()].copy()

# Aggregate to one row per customer
customers = df_customers_source.groupby('Customer ID').agg(
    first_purchase=('InvoiceDate', 'min'),
    last_purchase=('InvoiceDate', 'max'),
    total_orders=('Invoice', 'nunique'),
    total_items=('Quantity', 'sum'),
    total_revenue=('Revenue', 'sum'),
    country=('Country', lambda x: x.mode()[0])  # most common country for that customer
).reset_index()

# Customer lifespan in days
customers['lifespan_days'] = (customers['last_purchase'] - customers['first_purchase']).dt.days

# Average order value
customers['avg_order_value'] = (customers['total_revenue'] / customers['total_orders']).round(2)

# Cohort - the month of their first purchase
customers['cohort_month'] = customers['first_purchase'].dt.to_period('M').astype(str)

print(f"Customer dataset built: {len(customers):,} unique customers")
customers.head()

Customer dataset built: 5,878 unique customers


,Customer ID,first_purchase,last_purchase,total_orders,total_items,total_revenue,country,lifespan_days,avg_order_value,cohort_month
0,12346,2009-12-14 08:34:00,2011-01-18 10:01:00,12,74285,"77,556.46",United Kingdom,400,"6,463.04",2009-12
1,12347,2010-10-31 14:20:00,2011-12-07 15:52:00,8,2967,"4,921.53",Iceland,402,615.19,2010-10
2,12348,2010-09-27 14:59:00,2011-09-25 13:13:00,5,2714,"2,019.40",Finland,362,403.88,2010-09
3,12349,2010-04-29 13:20:00,2011-11-21 09:51:00,4,1624,"4,428.69",Italy,570,"1,107.17",2010-04
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,Norway,0,334.40,2011-02


In [20]:
# Make sure the output folder exists
import os
os.makedirs('../data/processed', exist_ok=True)

# If your notebook is in the same folder as the Excel file (no separate notebooks folder),
# use 'data/processed' instead of '../data/processed'

# Export both files
df_clean.to_csv('../data/processed/transactions_clean.csv', index=False)
customers.to_csv('../data/processed/customers_clean.csv', index=False)

print("Exported:")
print(f"  transactions_clean.csv  ({len(df_clean):,} rows)")
print(f"  customers_clean.csv     ({len(customers):,} rows)")

Exported:
  transactions_clean.csv  (1,007,913 rows)
  customers_clean.csv     (5,878 rows)


In [2]:
import pandas as pd
customers = pd.read_csv('customers_clean.csv')
print(f"Total customers: {len(customers):,}")
print(f"Customers with >1 order: {(customers['total_orders'] > 1).sum():,}")
print(f"Repeat rate: {(customers['total_orders'] > 1).mean():.1%}")

Total customers: 5,878
Customers with >1 order: 4,255
Repeat rate: 72.4%
